In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
path_train = '/kaggle/input/datasets/aithbb/starplatinum/train_data (7)/'
path_test = '/kaggle/input/datasets/aithbb/starplatinum/test_data (4)/'
print(os.listdir(path_train))
print(os.listdir(path_test))
train = pd.read_csv(path_train + 'train.csv')
# sc = StandardScaler()
# train['target_flux'] = sc.fit_transform(np.log1p(train[['target_flux']]))
o = 0
Xs = []
ys = []
train = train.sample(frac=1, random_state=42).reset_index(drop=True)
for image_id in tqdm(train['image_id']):
    X = plt.imread(path_train + 'train_images/' + image_id)
    ys.append(train['target_flux'].iloc[len(Xs) // 8])
    ys.append(train['target_flux'].iloc[len(Xs) // 8])
    ys.append(train['target_flux'].iloc[len(Xs) // 8])
    ys.append(train['target_flux'].iloc[len(Xs) // 8])
    ys.append(train['target_flux'].iloc[len(Xs) // 8])
    ys.append(train['target_flux'].iloc[len(Xs) // 8])
    ys.append(train['target_flux'].iloc[len(Xs) // 8])
    ys.append(train['target_flux'].iloc[len(Xs) // 8])
    Xs.append(X)
    X = np.rot90(X,1)
    Xs.append(X)
    X = np.rot90(X,1)
    Xs.append(X)
    X = np.rot90(X,1)
    Xs.append(X)
    X = np.fliplr(X)
    Xs.append(X)
    X = np.rot90(X,1)
    Xs.append(X)
    X = np.rot90(X,1)
    Xs.append(X)
    X = np.rot90(X,1)
    Xs.append(X)
Xs = np.array(Xs)
ys = np.array(ys)
td = len(Xs) // 4 * 3
X1 = Xs[:td]
X2 = Xs[td:]
y1 = ys[:td]
y2 = ys[td:]


['train_images', 'train.csv']
['test.csv', 'test_images']


  0%|          | 0/1000 [00:00<?, ?it/s]

In [2]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
traindataset = TensorDataset(torch.tensor(X1.reshape(-1, 1, 128, 128), dtype=torch.float32), torch.tensor(y1, dtype=torch.float32))
valdataset = TensorDataset(torch.tensor(X2.reshape(-1, 1, 128, 128)), torch.tensor(y2))
trainloader = DataLoader(traindataset, batch_size=32, shuffle=True)
valloader = DataLoader(valdataset, batch_size=32)
class Net(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d(1)
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.head(x)
        return x
model = Net().to('cuda')
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
best_val = 10**18
best_state = None
best_epoch = -1

for epoch in range(100):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data
        inputs = inputs.to('cuda')
        labels = labels.to('cuda')
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs.squeeze(1), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    model.eval()
    val_loss = 0
    preds = []
    trues = []
    
    with torch.no_grad():
        for inputs, labels in valloader:
            inputs = inputs.to('cuda')
            labels = labels.to('cuda')
            outputs = model(inputs)
    
            loss = criterion(
                outputs.squeeze(1),
                labels.float()
            )
    
            val_loss += loss.item()

            preds.append(outputs.squeeze(1).detach().cpu().numpy())
            trues.append(labels.float().detach().cpu().numpy())
    
    val_loss /= len(valloader)

    preds = np.concatenate(preds)
    trues = np.concatenate(trues)
    val_rmse = np.sqrt(mean_squared_error(trues, preds))

    if val_rmse < best_val:
        best_val = val_rmse
        best_epoch = epoch + 1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    
    print(
        f"Epoch {epoch+1} | "
        f"train={np.sqrt(running_loss/len(trainloader)):.4f} | "
        f"val={val_rmse:.4f} | "
        f"best_epoch={best_epoch} | "
        f"best_val={best_val:.4f}"
    )
    
    model.train()

if best_state is not None:
    model.load_state_dict(best_state)
    model = model.to('cuda')

print('best_epoch:', best_epoch, 'best_val:', best_val)
print('Finished Training')

Epoch 1 | train=6252.9468 | val=5992.2373 | best_epoch=1 | best_val=5992.2373
Epoch 2 | train=5783.6292 | val=5046.8870 | best_epoch=2 | best_val=5046.8870
Epoch 3 | train=4352.4323 | val=2668.1501 | best_epoch=3 | best_val=2668.1501
Epoch 4 | train=2507.9018 | val=1735.6093 | best_epoch=4 | best_val=1735.6093
Epoch 5 | train=1602.0530 | val=2833.8266 | best_epoch=4 | best_val=1735.6093
Epoch 6 | train=1512.0945 | val=908.0751 | best_epoch=6 | best_val=908.0751
Epoch 7 | train=1376.2707 | val=643.5282 | best_epoch=7 | best_val=643.5282
Epoch 8 | train=1284.2724 | val=2216.6218 | best_epoch=7 | best_val=643.5282
Epoch 9 | train=1192.2217 | val=607.7237 | best_epoch=9 | best_val=607.7237
Epoch 10 | train=1150.3722 | val=1206.0718 | best_epoch=9 | best_val=607.7237
Epoch 11 | train=1193.3797 | val=890.7439 | best_epoch=9 | best_val=607.7237
Epoch 12 | train=1101.1406 | val=767.6458 | best_epoch=9 | best_val=607.7237
Epoch 13 | train=903.5451 | val=582.7698 | best_epoch=13 | best_val=582.7

In [3]:
model.eval()

Xtest = []
test = pd.read_csv(path_test + 'test.csv')

with torch.no_grad():
    for image_id in tqdm(test['image_id']):
        img = plt.imread(path_test + 'test_images/' + image_id)

        if img.ndim == 3:
            img = img[..., 0]

        img = img.astype(np.float32)

        cur_preds = []
        cur = img

        for k in range(4):
            x = torch.tensor(cur.reshape(1, 1, 128, 128), dtype=torch.float32).to('cuda')

            p = model(x).detach().cpu().numpy()[0, 0]
            cur_preds.append(p)

            cur = np.rot90(cur, 1).copy()

        Xtest.append(np.mean(cur_preds))

pred = np.array(Xtest)
pred = np.clip(pred, 0, None)

  0%|          | 0/300 [00:00<?, ?it/s]

In [4]:
sub = pd.DataFrame({'subtaskID' : [1] * len(test) + [2] * len(test), 'datapointID' : list(test['image_id'])*2, 'answer' : ["(64 64)"] * len(test) + list(pred.reshape(-1))})
sub.to_csv('submission.csv', index=False)
sub

,subtaskID,datapointID,answer
0,1,00000.png,(64 64)
1,1,00001.png,(64 64)
2,1,00002.png,(64 64)
3,1,00003.png,(64 64)
4,1,00004.png,(64 64)
...,...,...,...
595,2,00295.png,4981.868652
596,2,00296.png,231.721649
597,2,00297.png,207.797226
598,2,00298.png,18986.785156
